# Exercise 1.3.1.5 — cross-dataset generalization matrix

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.3.1 Linear Probes`  
**Notebook:** `1.3.1_Linear_Probes_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.3.1.5](https://delta-drills.vercel.app/?arena_exercise=1.3.1.5)


# [1.3.1] Linear Probes (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/11_[1.3.1]_Linear_Probes)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part31_linear_probes/1.3.1_Linear_Probes_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part31_linear_probes/1.3.1_Linear_Probes_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/header-31.png" width="350">

# Introduction

This exercise set is built around **linear probing**, one of the most important tools in mechanistic interpretability for understanding what information language models represent internally.

We'll look at three papers:

- The [Geometry of Truth](https://arxiv.org/abs/2310.06824) paper by Marks & Tegmark, which shows that LLMs develop linear representations of truth that generalize across diverse datasets and are causally implicated in model outputs.
- The [deception probes paper](https://arxiv.org/abs/2502.03407) from Apollo Research, which extends this from factual truth to *strategic deception detection* - showing probes trained on simple contrastive data can generalize to realistic deception scenarios.
- The [high-stakes interactions paper](https://arxiv.org/abs/2506.10805) (NeurIPS 2025), which trains attention probes to detect whether a user's *request* is high-stakes - a different target from model intent - and shows they match full LLM classifiers at a fraction of the compute cost.

### What is probing?

The core idea: extract internal activations from a model, then train a simple classifier on them. If a *linear* probe can accurately classify some property from the activations, that property is **linearly represented** in the model's internal state.

From the Geometry of Truth paper:

> *"We identify a linear representation of truth that generalizes across several structurally and topically diverse datasets... these representations are not merely associated with truth, but are also causally implicated in the model's output."*

The "causally implicated" part matters a lot: it's not just that we can read off truth from model internals, but that the model actually *uses* these representations when computing outputs. We'll verify this in Section 3.

### Why this matters for safety

If we can reliably detect truth, deception, or intent from model internals, there are direct implications for model monitoring. [Neel Nanda argues](https://www.lesswrong.com/posts/G9HdpyREaCbFJjKu5/it-is-reasonable-to-research-how-to-use-model-internals-in) probes could even be used *during training*:

> *"There are certain things that may be much easier to specify using the internals of the model. For example: Did it do something for the right reasons? Did it only act this way because it knew it was being trained or watched?"*

But this is genuinely controversial. The worry, as [Bronson Schoen puts it](https://www.lesswrong.com/posts/G9HdpyREaCbFJjKu5/it-is-reasonable-to-research-how-to-use-model-internals-in?commentId=CtZnXwZuBgcWsagwn), is that training against probe signals might just teach the model to hide whatever the probe was measuring:

> *"If you train directly against non-obfuscated internals and no longer see bad behavior, the obvious possibility is that now you've just got obfuscated internals."*

For deception probes specifically, the generalization question is especially pointed: we may need to detect sophisticated deception in scenarios we've never seen. As the Apollo paper puts it: *"our monitors will need to exhibit generalization - correctly identifying deceptive text in new types of scenarios."*

### What you'll build

These exercises cover the full pipeline: extracting activations, visualizing with PCA, training probes, validating them causally, and applying them to deception detection. Layer choice, token position, and probe type all matter - part of the point is developing intuition for *why*.

### Models we'll use

Sections 1-3 use `meta-llama/Llama-2-13b-hf` (base model, ~26GB in bfloat16). The Geometry of Truth paper has specific configurations for this model (`probe_layer=14`, `intervene_layer=8`), so our results should closely match theirs. Section 4 switches to `meta-llama/Meta-Llama-3.1-8B-Instruct` (instruct-tuned, ~16GB), needed for the deception detection instructed-pairs methodology.

Both fit comfortably on a single A100. If you have a multi-GPU setup, the 70B variants are worth trying as a bonus; the paper's strongest results are at that scale.

## Content & Learning Objectives

### 1️⃣ Setup & visualizing truth representations

> ##### Learning Objectives
>
> * Extract hidden state activations from specified layers and token positions
> * Implement PCA to visualize high-dimensional activations
> * Observe that truth is linearly separable in activation space - even without supervision
> * Understand which layers best represent truth via a layer sweep

### 2️⃣ Training & comparing probes

> ##### Learning Objectives
>
> * Implement difference-of-means (MM) and logistic regression (LR) probes
> * Compare probe types: accuracy, direction similarity, and what each captures
> * Understand CCS (Contrastive Consistent Search) as an unsupervised alternative and its limitations

### 3️⃣ Causal interventions

> ##### Learning Objectives
>
> * Understand why classification accuracy alone is insufficient - causal evidence is needed
> * Implement activation patching with probe directions to flip model predictions
> * Compare the causal effects of MM vs. LR probe directions
> * Appreciate that MM probes find more causally implicated directions despite lower classification accuracy

### 4️⃣ Probing for Deception

> ##### Learning Objectives
>
> * Construct instructed-pairs datasets following the deception-detection paper's methodology
> * Train deception probes on instruct-tuned models
> * Evaluate whether deception probes generalize to factual truth/falsehood datasets
> * Understand methodological choices that affect replicability

### 5️⃣ Attention Probes for High-Stakes Detection

> ##### Learning Objectives
>
> * Understand what "high-stakes interactions" means as a probe target, and why it differs from probing model intent
> * Extract full-sequence activations (shape `(n, seq, d_model)`) rather than last-token only
> * Implement an attention probe as a `nn.Module` - a single learned query that computes a weighted sum over token positions before classification
> * Compare attention pooling against last-token and mean-pool baselines using AUROC
> * Inspect learned attention weights to understand which parts of a prompt are most diagnostic

## Reading Material

The core papers we'll be replicating are "The Geometry of Truth" and "Detecting Strategic Deception Using Linear Probes" - you should at least skim both before starting, so you understand the basics. The other references give you more context on the probing literature and the open questions around it.

- [The Geometry of Truth: Emergent Linear Structure in Large Language Model Representations of True/False Datasets](https://arxiv.org/abs/2310.06824) by Marks & Tegmark (COLM 2024). Shows that LLMs develop linear representations of truth that generalise across diverse datasets and are causally implicated in model outputs. Read at least the abstract, and sections 1, 2 & 4 (intro, datasets & visualization). You can skip 3, because we won't be doing much of this kind of patching in these exercises.
- [Detecting Strategic Deception Using Linear Probes](https://arxiv.org/abs/2502.03407) by Goldowsky-Dill et al. (Apollo Research, 2025). Extends truth probing to *strategic deception detection*, showing that probes trained on simple contrastive data can generalise to realistic deception scenarios. Section 4 of this exercise set replicates their methodology. Read the abstract and sections 1 & 3 (introduction and methodology).
- [Detecting High-Stakes Interactions with Activation Probes](https://arxiv.org/abs/2506.10805) by McKenzie et al. (NeurIPS 2025). Trains attention probes to detect whether a user's request is high-stakes, matching full LLM classifiers at much lower cost. Section 5 of this exercise set replicates this. Read the abstract and section 2 (methodology), or alternatively just the [EleutherAI post](https://blog.eleuther.ai/attention-probes/).
- [Discovering Latent Knowledge in Language Models Without Supervision](https://arxiv.org/abs/2212.03827) by Burns et al. (ICLR 2023). Introduces Contrastive Consistent Search (CCS), an unsupervised method for finding truth directions without labelled data. We discuss CCS limitations in section 2 but don't implement it in full. Optional reading.

## Setup code

Before running this, you'll need to clone the Geometry of Truth as well as Deception Detection repos into the `exercises` directory:

```bash
cd chapter1_transformer_interp/exercises

git clone https://github.com/saprmarks/geometry-of-truth.git
git clone https://github.com/ApolloResearch/deception-detection.git
```

`Llama-2-13b-hf` is a gated model, so you'll need a HuggingFace access token (as well as requesting access [here](https://huggingface.co/meta-llama/Llama-2-13b-hf)). When you've got access and made a HuggingFace token, create a `.env` file in your `chapter1_transformer_interp/exercises` directory with:

```
HF_TOKEN=hf_your_token_here
```

then the code below will use this token for authentication.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install "openai==1.56.1" einops datasets jaxtyping "sae-lens>=4.0.0,<5.0.0" openai tabulate umap-learn hdbscan eindex-callum git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python git+https://github.com/callummcdougall/sae_vis.git@callum/v3 transformer_lens==2.17.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import gc
import json
import os
import pickle
import sys
from dataclasses import dataclass
from pathlib import Path

import circuitsvis as cv
import einops
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from IPython.display import HTML, display
from jaxtyping import Bool, Float
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

device = t.device("cuda" if t.cuda.is_available() else "cpu")
dtype = t.bfloat16

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part31_linear_probes"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part31_linear_probes.tests as tests
import part31_linear_probes.utils as utils

MAIN = __name__ == "__main__"

In [ ]:
# Set up paths to the cloned repos
# Adjust these if your repos are in a different location
GOT_ROOT = exercises_dir / "geometry-of-truth"  # geometry-of-truth repo
DD_ROOT = exercises_dir / "deception-detection"  # deception-detection repo

assert GOT_ROOT.exists(), f"Please clone geometry-of-truth repo to {GOT_ROOT}"
assert DD_ROOT.exists(), f"Please clone deception-detection repo to {DD_ROOT}"

GOT_DATASETS = GOT_ROOT / "datasets"
DD_DATA = DD_ROOT / "data"

### Loading the model

We start with LLaMA-2-13B, a base (not instruction-tuned) model. The Geometry of Truth paper uses this model with `probe_layer=14` and `intervene_layer=8` - we'll use these exact values.

In [ ]:
load_dotenv(dotenv_path=str(exercises_dir / ".env"))
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "Please set HF_TOKEN in your chapter1_transformer_interp/exercises/.env file"

In [ ]:
MODEL_NAME = "meta-llama/Llama-2-13b-hf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map="auto",
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

NUM_LAYERS = len(model.model.layers)
D_MODEL = model.config.hidden_size
# Layer choices from the geometry-of-truth repo config for llama-2-13b. The paper
# found truth representations are concentrated in early-to-mid layers, and identified
# these specific layers via patching experiments (Section 3, "group (b)").
PROBE_LAYER = 14
INTERVENE_LAYER = 8

print(f"Model: {MODEL_NAME}")
print(f"Layers: {NUM_LAYERS}, Hidden dim: {D_MODEL}")
print(f"Probe layer: {PROBE_LAYER}, Intervene layer: {INTERVENE_LAYER}")

### Loading the datasets

The Geometry of Truth paper uses several carefully curated datasets of simple true/false statements. Each dataset has a `statement` column and a `label` column (1=true, 0=false).

From the paper:
> *"We find that the truth-related structure in LLM representations is much cleaner for our curated datasets than for our unstructured ones."*

Let's load three of these curated datasets and examine them:

In [ ]:
DATASET_NAMES = ["cities", "sp_en_trans", "larger_than"]

datasets = {}
for name in DATASET_NAMES:
    df = pd.read_csv(GOT_DATASETS / f"{name}.csv")
    datasets[name] = df
    print(f"\n{name}: {len(df)} statements ({df['label'].sum()} true, {(1 - df['label']).sum():.0f} false)")
    display(df.head(4))

# 1️⃣ Setup & visualizing truth representations

> ##### Learning Objectives
>
> * Extract hidden state activations from specified layers and token positions
> * Implement PCA to visualize high-dimensional activations
> * Observe that truth is linearly separable in activation space - even without supervision
> * Understand which layers best represent truth via a layer sweep

## Extracting activations

Our first task is to extract hidden state activations from the model. For the Geometry of Truth approach, we extract the **last-token** activation at each specified layer. For declarative statements like "The city of Paris is in France.", the model's representation of whether the statement is true or false is concentrated at the final token position.

Note - the Geometry of Truth paper probes specifically at the **end-of-sentence punctuation** token (the period / full stop). The datasets are designed so that every statement ends with a period, meaning the last token is always the period. This is important because the model's truth representation builds up over the sentence and is concentrated at the final punctuation mark.

A few technical details to keep in mind. We use `output_hidden_states=True` in the forward pass to get all layer activations. `outputs.hidden_states` has length `num_layers + 1`: index 0 is the embedding output, and index `i` for `i >= 1` is the output of layer `i-1`. We also need to handle **padding** correctly, since statements have different lengths. We pad them but must extract the activation at the last *real* (non-padding) token, not the last position.

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.3.1.5"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part31_linear_probes.solutions import extract_activations, get_pca_components, layer_sweep_accuracy, MMProbe, LRProbe


### Exercise - cross-dataset generalization matrix

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 15-20 minutes on this exercise.
> Cross-dataset generalization is the real test of whether your probe has found a universal truth direction.
> ```

Train MM and LR probes on each of the 3 curated datasets and evaluate each probe on all 3 datasets, producing 3×3 accuracy matrices. If the model has a unified truth direction, off-diagonal entries should be high. The Geometry of Truth paper reports strong cross-generalization for LR probes at 13B scale, and even stronger at 70B:

> *"The LR probes transfer almost perfectly: probes trained on any one of our three datasets achieve near-ceiling accuracy on the other two."*

Do your results agree? You should also compute pairwise cosine similarities between probe directions trained on different datasets to check whether they point in a similar direction.

In [ ]:
def compute_generalization_matrix(
    train_acts: dict[str, Float[Tensor, "n d"]],
    train_labels: dict[str, Float[Tensor, " n"]],
    test_acts: dict[str, Float[Tensor, "n d"]],
    test_labels: dict[str, Float[Tensor, " n"]],
    dataset_names: list[str],
    probe_cls: type,
) -> Float[Tensor, "n_datasets n_datasets"]:
    """
    Compute a generalization matrix: entry (i, j) is the test accuracy of a probe trained on dataset i
    and evaluated on dataset j.

    Args:
        train_acts, train_labels: Training data per dataset.
        test_acts, test_labels: Test data per dataset.
        dataset_names: Names of datasets (determines matrix ordering).
        probe_cls: Probe class to use (MMProbe or LRProbe), must have from_data and pred methods.

    Returns:
        Tensor of shape [n_datasets, n_datasets] with accuracy values.
    """
    raise NotImplementedError()


mm_matrix = compute_generalization_matrix(train_acts, train_labels, test_acts, test_labels, DATASET_NAMES, MMProbe)
lr_matrix = compute_generalization_matrix(train_acts, train_labels, test_acts, test_labels, DATASET_NAMES, LRProbe)

assert mm_matrix.shape == (3, 3), f"Wrong shape: {mm_matrix.shape}"
assert (mm_matrix.diag() > 0.6).all(), "In-distribution accuracy should be at least 60%"

# Heatmap visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=["MMProbe", "LRProbe"], horizontal_spacing=0.15)

for idx, (matrix, name) in enumerate([(mm_matrix, "MM"), (lr_matrix, "LR")]):
    text_vals = [[f"{matrix[i, j]:.3f}" for j in range(len(DATASET_NAMES))] for i in range(len(DATASET_NAMES))]
    fig.add_trace(
        go.Heatmap(
            z=matrix.numpy(),
            x=DATASET_NAMES,
            y=DATASET_NAMES,
            text=text_vals,
            texttemplate="%{text}",
            colorscale="RdYlGn",
            zmin=0.5,
            zmax=1.0,
            showscale=(idx == 1),
        ),
        row=1,
        col=idx + 1,
    )
    fig.update_yaxes(title_text="Train dataset" if idx == 0 else "", row=1, col=idx + 1)
    fig.update_xaxes(title_text="Test dataset", row=1, col=idx + 1)

fig.update_layout(title="Cross-dataset Generalization (Test Accuracy)", height=400, width=800)
fig.show()

# Cosine similarity between probe directions
mm_directions = {name: MMProbe.from_data(train_acts[name], train_labels[name]).direction for name in DATASET_NAMES}
lr_directions = {name: LRProbe.from_data(train_acts[name], train_labels[name]).direction for name in DATASET_NAMES}

print("\nPairwise cosine similarity between probe directions:")
for probe_name, directions in [("MM", mm_directions), ("LR", lr_directions)]:
    print(f"\n  {probe_name}Probe:")
    for i, n1 in enumerate(DATASET_NAMES):
        for j, n2 in enumerate(DATASET_NAMES):
            if j > i:
                d1 = directions[n1] / directions[n1].norm()
                d2 = directions[n2] / directions[n2].norm()
                print(f"    {n1} vs {n2}: {(d1 @ d2).item():.4f}")

<details>
<summary>Discussion - interpreting the generalization matrix</summary>

If the model has a single "truth direction" that is shared across domains, then probes trained on *any* dataset should achieve high accuracy on *all* datasets (high off-diagonal entries). If instead the model represents truth in domain-specific ways, you'll see high diagonal (in-distribution) but low off-diagonal (out-of-distribution) accuracy.

The Geometry of Truth paper reports that LR probes generalize almost perfectly at 13B+ scale, which suggests a largely unified truth representation. However, at smaller model scales or for very different domains, you may see more variation.

As a further test, try evaluating your cities probe on the `likely` dataset from the geometry-of-truth repo, which contains nonfactual text with likely or unlikely continuations (code snippets, random text). If the truth probe also separates this dataset, the probe may detect *text probability* rather than *factual truth*. The paper predicts (and confirms) that a well-trained truth probe should **not** separate `likely`, because truth and likelihood are distinct features.
</details>


<details><summary>Solution</summary>

```python
def compute_generalization_matrix(
    train_acts: dict[str, Float[Tensor, "n d"]],
    train_labels: dict[str, Float[Tensor, " n"]],
    test_acts: dict[str, Float[Tensor, "n d"]],
    test_labels: dict[str, Float[Tensor, " n"]],
    dataset_names: list[str],
    probe_cls: type,
) -> Float[Tensor, "n_datasets n_datasets"]:
    """
    Compute a generalization matrix: entry (i, j) is the test accuracy of a probe trained on dataset i
    and evaluated on dataset j.

    Args:
        train_acts, train_labels: Training data per dataset.
        test_acts, test_labels: Test data per dataset.
        dataset_names: Names of datasets (determines matrix ordering).
        probe_cls: Probe class to use (MMProbe or LRProbe), must have from_data and pred methods.

    Returns:
        Tensor of shape [n_datasets, n_datasets] with accuracy values.
    """
    n = len(dataset_names)
    matrix = t.zeros(n, n)
    for i, train_name in enumerate(dataset_names):
        probe = probe_cls.from_data(train_acts[train_name], train_labels[train_name])
        for j, test_name in enumerate(dataset_names):
            preds = probe.pred(test_acts[test_name])
            acc = (preds == test_labels[test_name]).float().mean().item()
            matrix[i, j] = acc
    return matrix
```
</details>

<details>
<summary>A note about CCS (optional)</summary>

## Contrastive Consistent Search (CCS) and the unsupervised discovery debate

Before moving to causal interventions, it's worth discussing **CCS** (Contrastive Consistent Search), which represents one of the most exciting and debated ideas in probing research.

### The CCS method

CCS is **unsupervised**: it requires paired positive/negative statements but **no labels**. The loss enforces that `p(x) ≈ 1 - p(neg_x)` (consistency) and that the probe is confident. This suggests truth might be discoverable without any supervision.

### Key critiques

**Contrast pairs do all the work.** [Emmons (2023)](https://www.lesswrong.com/posts/9vwekjD6xyuePX7Zr/contrast-pairs-drive-the-empirical-performance-of-contrast) shows that PCA on contrast pair differences achieves 97% of CCS's accuracy. The CCS loss function is largely redundant; the contrast pair construction is the real innovation.

**CCS may not find knowledge.** [Farquhar et al. (Google DeepMind, 2023)](https://www.lesswrong.com/posts/wtfvbsYjNHYYBmT3k/discussion-challenges-with-unsupervised-llm-knowledge-1) prove that for *every* possible binary classification, there exists a zero-loss CCS probe, so the loss has no structural preference for knowledge over arbitrary features. When they append random distractor words to contrast pairs, CCS learns to classify the distractor instead of truth.

**The XOR problem.** [Sam Marks (2024)](https://www.lesswrong.com/posts/hjJXCn9GsskysDceS/what-s-up-with-llms-representing-xors-of-arbitrary-features) shows that LLMs linearly represent XORs of arbitrary features, creating a new failure mode for probes: a probe could learn `truth XOR geography` which works on geographic data but fails under distribution shift. Empirically probes often *do* generalize, likely because basic features have higher variance than their XORs, but this is a matter of degree, not a guarantee. (Note, most of this paper talks about the XOR problem in a broader way than just its relevance to CCS - still strongly recommended as further reading!)

### Takeaway

The CCS literature tells a cautionary tale: probing accuracy on a single dataset tells you very little. This is why Section 3 demands **causal evidence** via interventions, and why cross-dataset generalization is the real test of a probe.

### Bonus exercise: implement CCS

If you want to dig deeper, try implementing CCS yourself using the `neg_cities` dataset (which provides natural contrast pairs). The core idea: take the difference vector for each contrast pair (true statement activation minus false statement activation), then run PCA on those difference vectors. As Emmons shows, just the first principal component of the difference vectors recovers ~97% of CCS's accuracy, so you don't even need the CCS loss. You can then compare this unsupervised direction to your supervised MM and LR probes on the generalization datasets from Section 2, and see how the causal effects compare in Section 3.

For further reading: Levinstein & Herrmann (2024), "Still No Lie Detector for Language Models"; the geometry-of-truth repo includes a full `CCSProbe` implementation in `probes.py`.

</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
